# Melbourne Suburb Analytics

## Section 2: Analysis

## 2.1 Setup

### 2.1.1 Configure Libraries

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

In [2]:
pd.set_option('display.max_columns', None)

---

### 2.1.2 Load Dataset

In [3]:
proj_root = Path.cwd().parent
df_raw = pd.read_csv(proj_root/"data"/"processed"/"melb_data_cleaned.csv")

df = df_raw.copy()

<sub>◦ *This notebook uses exported csv from [01_cleaning.ipynb](./01_cleaning.ipynb), not in-memory state*</sub>

---

## 2.2 Analysis

### 2.2.1 Does average price differ significantly between two property types?

The two property types we will be comparing are **houses** `h` and **units** `u`

Meaning: do houses actually sell for more compared to units, or is this information just luck of the chosen properties from this dataset?

#### Hypothesis
Null Hypothesis ($H_0$): The average prices are the same, where any observed difference in the sample is due to random sampling (chance)  
- $H_0$: $μ_h = μ_u$  

Alternative Hypothesis ($H_1$): The average prices are the different, where there is a genuine difference between the **population** mean house price and the **population** mean unit price
- $H_1$: $μ_h \neq μ_u$

---

#### Descriptive Statistics
We identify the sample mean (x̄), standard deviation (s), sample size (n) for houses and units, and calculate the mean ratio.

In [4]:
summary = df.groupby("Type")["Price"].agg(["mean", "std", "count"])

summary.style.format("{:.2f}") ## .2f for display

,mean,std,count
Type,,,
h,1242664.76,668078.74,9449.00
t,933735.05,395038.25,1114.00
u,605127.48,260987.45,3017.00


In [5]:
summary.loc["h", "mean"]/summary.loc["u", "mean"]

np.float64(2.0535586182904413)

For houses and units (Price):  
`h` :  
sample mean (x̄) = 1242664.76  
standard deviation (s) = 668078.74  
sample size (n) = 9449  

`u` :  
sample mean (x̄) = 605127.48	  
standard deviation (s) = 260987.45  
sample size (n) = 3017 

- Houses sell for roughly **2.05x** the price of units on average

<sub>**rounded to 2dp* </sub>  

---

#### Standard Error
We find the **standard error** of the difference between two means using:
$$
SE = \sqrt{\frac{s_h^2}{n_h} + \frac{s_u^2}{n_u}}
$$
- SE shrinks when: sample size is large or there is less noise
- SE grows when: sample size is small or data is spread out

In [6]:
s_h = summary["std"]["h"]
n_h = summary["count"]["h"]
s_u = summary["std"]["u"]
n_u = summary["count"]["u"]
se = np.sqrt((s_h**2/n_h)+(s_u**2/n_u))
print(se)

8355.386492517506


Standard Error (SE) of the difference in mean prices between groups `h` and `u` = **~8355.38**

---

#### t-statistic
We now compute the **t-statistic** to compare our calculated SE using:
$$
t = \frac{\bar{x}_h - \bar{x}_u}{SE}
$$
- This shows (the observed difference in means) ÷ (difference between two sample means expected from random sampling)
- A larger (absolute) t-statistic = the observed difference is **unlikely to be explained just by random sampling**
    - observed difference is **most likely to continue** if the sampling process were to be repeated
    - observed difference is **many standard errors away from 0**
- A t-statistic close to 0 = the observed difference could **possibly be explained by random sampling**
    - observed difference may be **less stable** and could change from sample to sample by chance
    - observed difference is **a few standard errors away from 0**

In [7]:
difference = summary["mean"]["h"]-summary["mean"]["u"]
difference

np.float64(637537.2765514065)

In [8]:
t_stat = difference/se
print(t_stat)

76.30254771844963


t-statistic = ~76.3
- The t-statistic is very large in magnitude, indicating that the observed difference in mean prices between houses and units is extremely large, relative to the estimated sampling uncertainty
- This shows that the difference in mean prices are highly stable and are likely to remain meaningful if the sampling process were repeated

---

#### Degrees of freedom
We must calculate the degrees of freedom (df) in order to find the p-value.
- df determines which t-distribution to be used when interpreting the t-statistic
- Conceptually, df represents the amount of information avaliable for estimating uncertainty from the sample data
    - more information = reduces uncertainty in the estimated sampling distribution
    - larger sample sizes provide more information about the populations
- As df increases, the t-distribution becomes more similar to the standard **normal** distribution

Since the house and unit groups have **different sample sizes and variances**, the Welch-Satterthwaite approximation is used to estimate df:
$$
df
=
\frac{
\left(
\frac{s_h^2}{n_h}
+
\frac{s_u^2}{n_u}
\right)^2
}
{
\frac{
\left(\frac{s_h^2}{n_h}\right)^2
}{n_h-1}
+
\frac{
\left(\frac{s_u^2}{n_u}\right)^2
}{n_u-1}
}
$$

In [9]:
div_h = s_h**2/n_h
div_u = s_u**2/n_u
top = (div_h+div_u)**2
bottom_left = ((div_h)**2)/(n_h-1)
bottom_right = ((div_u)**2)/(n_u-1)
d_f = top/(bottom_left+bottom_right)
print(d_f)

12029.283596165345


Degrees of freedom = ~12029
- This means we have a very large amount of information avaliable for estimating the sampling distribution
    - Hence, the corresponding t-distrubtion is **extremely close** to the standard distribution
    - This means small uncertainty with estimating the sampling distribution

---

#### p-value
We now convert this into a **p-value** via the t-statistic.
- p-value represents the probability of observing a t-statistic at least as extreme as the one obtained, assuming the null hypothesis ($H_0$) is true
- df determines the appropiate t-distrubtion used in this calculation

- A smaller p-value = the observed difference would be **unlikely** if the population mean prices were actually equal
- A large p-value = the observed difference is **reasonably consistent** with random sampling variation if the population mean prices were equal

For a **two-tailed Welch t-test**, the p-value is:
$$
p = 2 \times P(T \ge |t|)
$$

where:

- $T$ utilizes a t-distribution with the calculated degrees of freedom.
- $|t|$ is the absolute value of the calculated t-statistic.
- We multiply by 2 to account for both tails of the distribution, since we are testing for **any difference** in mean price rather than a specific direction.

In [10]:
p_val = 2 * stats.t.sf(abs(t_stat), d_f)
p_val

np.float64(0.0)

p-value = ~0 (so close to 0 that it was reported by Python as `0.0`)

Since the p-value is substantially smaller than the significant level of 0.05, we reject the null hypothesis.  
There is very strong statistical evidence that the average price differs between houses and units.

---

### Verification with SciPy

In [15]:
house_prices = df.loc[df["Type"] == "h", "Price"]
unit_prices = df.loc[df["Type"] == "u", "Price"]

result = stats.ttest_ind(
    house_prices,
    unit_prices,
    equal_var=False
)
print(f"t-statistic = {result.statistic}, p-value = {result.pvalue}, degrees of freedom = {result.df}")

t-statistic = 76.30254771844955, p-value = 0.0, degrees of freedom = 12029.283596165349


To verify the manual calculations, I used SciPy's `stats.test_ind()` function to perform Welch's two-sample t-test.

The test returned:
- t-statistic = ~76.3
- degrees of freedom = ~12029
- pvalue = ~0

These value match the manually calculated t-statistic, degrees of freedom and p-value which confirms that the calculations performed above are correct.

---

### Conclusion

The extremely large t-statistic (76.3) shows that the observed difference in mean prices between houses and units is very large relative to the estimated sampling uncertainty.

The p-value was virtually 0, meaning that the observed difference would be very unlikely if the population mean house price and population mean unit price were equal.

Therefore, we **reject** the null hypothesis ($H_0$).
There is **very strong statistical evidence** that **the average property price differs between houses and units**.

---